# SXGA2: Setup (run once, as ACCOUNTADMIN)

Creates the isolated environment the lesson notebooks run in:

| Course object | Isolated stand-in | Notes |
|---|---|---|
| ACCOUNTADMIN role | `SXGA2_LLM_ADMIN` | holds the account privileges the course uses as ACCOUNTADMIN |
| SYSADMIN role | `SXGA2_LLM_ROLE` | granted to `SXGA2_LLM_ADMIN`, mirroring SYSADMIN under ACCOUNTADMIN |
| COMPUTE_WH | `SXGA2_LLM_WH` | X-Small, auto-suspend, owned by `SXGA2_LLM_ROLE` |
| UTIL_DB | `UTIL_DB` | shared across badges; owned by SYSADMIN |
| learner's Downloads folder | `@UTIL_DB.PUBLIC.SXGA2_LLM_DOWNLOADS` | holds the lyric PDFs the course has learners download |
| learner's API Ninjas key | `UTIL_DB.PUBLIC.SXGA2_API_NINJAS_KEY` | a Snowflake secret. The key is never written into a notebook. |

Create the secret once, outside these notebooks, with your own key from api-ninjas.com:

```sql
use role accountadmin;
create or replace secret UTIL_DB.PUBLIC.SXGA2_API_NINJAS_KEY type = generic_string secret_string = '<your API Ninjas key>';
```

Course objects (SXGA2, DOC_AI_ROLE, DOC_AI_WH, DOC_AI_DB, the API integration, ...) keep the course's exact names and are created in the lessons.

In [ ]:
# Isolation mapping (set once per notebook; SQL cells use these via {{...}})
ADMIN_ROLE    = "SXGA2_LLM_ADMIN"   # stands in for the course's ACCOUNTADMIN
SYSADMIN_ROLE = "SXGA2_LLM_ROLE"    # stands in for the course's SYSADMIN
WAREHOUSE     = "SXGA2_LLM_WH"      # stands in for the course's COMPUTE_WH


def pdf(result):
    """A SQL cell's result as pandas (runtime 2.6+ returns a Snowpark DataFrame)."""
    return result.to_pandas() if hasattr(result, "to_pandas") else result

## Check for name collisions
Course object names are account-wide.

In [ ]:
%%sql -r existing_course_objects
use role accountadmin;
show databases like 'SXGA2';
show databases like 'DOC_AI_DB';
show roles like 'DOC_AI_ROLE';
show warehouses like 'DOC_AI_WH';
show integrations like 'API_NINJAS_ACCESS_INTEGRATION';

## Roles and grants

In [ ]:
%%sql -r create_roles
use role accountadmin;
create role if not exists {{ADMIN_ROLE}} comment = 'SXGA2 stand-in for ACCOUNTADMIN';
create role if not exists {{SYSADMIN_ROLE}} comment = 'SXGA2 stand-in for SYSADMIN';
grant role {{SYSADMIN_ROLE}} to role {{ADMIN_ROLE}};
grant role {{ADMIN_ROLE}} to role sysadmin;
set badge_user = current_user();
grant role {{ADMIN_ROLE}} to user identifier($badge_user);
grant role {{SYSADMIN_ROLE}} to user identifier($badge_user);
-- SYSADMIN creates the SXGA2 database (Lesson 3)
grant create database on account to role {{SYSADMIN_ROLE}};
-- the course's ACCOUNTADMIN steps: external access integration (Lesson 3)
grant create integration on account to role {{ADMIN_ROLE}};
-- AI functions
grant database role snowflake.cortex_user to role {{SYSADMIN_ROLE}};
grant database role snowflake.cortex_user to role {{ADMIN_ROLE}};
grant database role snowflake.ai_functions_user to role {{SYSADMIN_ROLE}};
grant database role snowflake.ai_functions_user to role {{ADMIN_ROLE}};
grant imported privileges on database snowflake to role {{ADMIN_ROLE}};

## Warehouse (stands in for COMPUTE_WH)

In [ ]:
%%sql -r create_warehouse
use role accountadmin;
create warehouse if not exists {{WAREHOUSE}}
  warehouse_size = xsmall auto_suspend = 60 auto_resume = true initially_suspended = true
  comment = 'SXGA2 stand-in for COMPUTE_WH';
grant ownership on warehouse {{WAREHOUSE}} to role {{SYSADMIN_ROLE}} copy current grants;

## UTIL_DB (shared across badges) and the API key secret

In [ ]:
%%sql -r util_db_grants
use role accountadmin;
create database if not exists UTIL_DB comment = 'Shared utility database for badge course audits (DORA grader, course stages)';
grant ownership on database UTIL_DB to role sysadmin copy current grants;
grant ownership on schema UTIL_DB.PUBLIC to role sysadmin copy current grants;
grant usage on database UTIL_DB to role {{ADMIN_ROLE}};
grant usage on database UTIL_DB to role {{SYSADMIN_ROLE}};
grant usage, create stage, create workspace on schema UTIL_DB.PUBLIC to role {{ADMIN_ROLE}};
grant usage on schema UTIL_DB.PUBLIC to role {{SYSADMIN_ROLE}};
-- the function in Lesson 3 reads the API key from this secret, so its owner role needs usage
grant read on secret UTIL_DB.PUBLIC.SXGA2_API_NINJAS_KEY to role {{ADMIN_ROLE}};
grant read on secret UTIL_DB.PUBLIC.SXGA2_API_NINJAS_KEY to role {{SYSADMIN_ROLE}};
grant usage on secret UTIL_DB.PUBLIC.SXGA2_API_NINJAS_KEY to role {{ADMIN_ROLE}};
grant usage on secret UTIL_DB.PUBLIC.SXGA2_API_NINJAS_KEY to role {{SYSADMIN_ROLE}};

## Course data files (the learner's Downloads folder)
The lyric PDFs are in `data/` beside these notebooks; upload them once:

```
snow sql -q "use role SXGA2_LLM_ADMIN; put 'file://<path>/data/*' @UTIL_DB.PUBLIC.SXGA2_LLM_DOWNLOADS auto_compress=false overwrite=true"
```

In [ ]:
%%sql -r create_downloads_stage
use role {{ADMIN_ROLE}};
create stage if not exists UTIL_DB.PUBLIC.SXGA2_LLM_DOWNLOADS
  comment = 'SXGA2: stands in for the learner''s local Downloads folder'
  directory = (enable = true)
  encryption = (type = 'SNOWFLAKE_SSE');
grant read on stage UTIL_DB.PUBLIC.SXGA2_LLM_DOWNLOADS to role {{SYSADMIN_ROLE}};
grant read on stage UTIL_DB.PUBLIC.SXGA2_LLM_DOWNLOADS to role {{ADMIN_ROLE}};
alter stage UTIL_DB.PUBLIC.SXGA2_LLM_DOWNLOADS refresh;
list @UTIL_DB.PUBLIC.SXGA2_LLM_DOWNLOADS;

In [ ]:
expected = ['99 Luftballons by Nena.pdf', '99 Red Balloons by Nena.pdf', 'Beyond the Sea by Bobby Darin.pdf', 'Blood Sweat & Tears by BTS JA.pdf', 'Blood Sweat & Tears by BTS KO.pdf', 'Cutie Pie by One Way.pdf', 'Flashdance... What a Feeling by Irene Cara.pdf', 'Flashdance... vilken känsla by Ingmar Nordströms.pdf', 'Fortnight by Taylor Swift.pdf', 'Garota de Ipanema by Antonio Carlos Jobim.pdf', 'Gloria by Laura Branigan.pdf', 'Gloria by Umberto Tozzi.pdf', 'La Mer by Charles Trénet.pdf', 'Mary Had A Little Lamb by Storybooksong.pdf', 'Ribbon in the Sky by Stevie Wonder.pdf', 'September by Earth, Wind & Fire.pdf', 'The Girl From Ipanema by Amy Winehouse.pdf', '_3 by Bad Bunny.pdf']
found = sorted(n.split('/')[-1] for n in pdf(create_downloads_stage)['name'])
print('Missing course files:', sorted(set(expected) - set(found)) or 'none')

## Teardown (commented out)

In [ ]:
%%sql -r teardown
-- use role accountadmin;
-- drop database if exists SXGA2;
-- drop database if exists DOC_AI_DB;
-- drop warehouse if exists DOC_AI_WH;
-- drop role if exists DOC_AI_ROLE;
-- drop integration if exists API_NINJAS_ACCESS_INTEGRATION;
-- drop stage if exists UTIL_DB.PUBLIC.SXGA2_LLM_DOWNLOADS;
-- drop warehouse if exists {{WAREHOUSE}};
-- drop role if exists {{SYSADMIN_ROLE}};
-- drop role if exists {{ADMIN_ROLE}};
select 'teardown is commented out' as status;